In [6]:
import requests
import pandas as pd

In [4]:
df = pd.read_csv('books.csv')

In [5]:
df.head()

,title,book_url,pdf_url
0,Think Python 2e,https://greenteapress.com/wp/think-python-2e/,http://greenteapress.com/thinkpython2/thinkpyt...
1,Think DSP,https://greenteapress.com/wp/think-dsp/,http://greenteapress.com/thinkdsp/thinkdsp.pdf
2,Think Complexity 2e,https://greenteapress.com/wp/think-complexity/,http://greenteapress.com/complexity2/thinkcomp...
3,Think Java 2e,https://greenteapress.com/wp/think-java-2e/,http://greenteapress.com/thinkjava7/thinkjava2...
4,Physical Modeling in MATLAB,https://greenteapress.com/wp/physical-modeling...,https://github.com/AllenDowney/PhysicalModelin...


# Download data and convert to markdown files

In [2]:
import csv
import os
import requests
from markitdown import MarkItDown

CSV_FILE = "books.csv"
PDF_DIR = "books"
MD_DIR = "books_text"

os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(MD_DIR, exist_ok=True)

def download_file(url, output_path):
    if os.path.exists(output_path):
        print(f"Already exists: {output_path}")
        return

    try:
        response = requests.get(url, stream=True, timeout=30)
        response.raise_for_status()

        with open(output_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        print(f"Downloaded: {output_path}")

    except requests.exceptions.RequestException as e:
        print(f"Failed to download {url}: {e}")

def convert_pdf_to_md(pdf_path, md_path, md_converter):
    try:
        result = md_converter.convert(pdf_path)
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(result.text_content)

        print(f"Converted: {md_path}")

    except Exception as e:
        print(f"Failed to convert {pdf_path}: {e}")

md = MarkItDown()

with open(CSV_FILE, newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)

    for row in reader:
        title = row["title"]
        pdf_url = row["pdf_url"]

        safe_name = title.replace(" ", "_").replace("/", "_")

        pdf_path = os.path.join(PDF_DIR, safe_name + ".pdf")
        md_path = os.path.join(MD_DIR, safe_name + ".md")

        # Step 1: Download
        download_file(pdf_url, pdf_path)

        # Step 2: Convert
        convert_pdf_to_md(pdf_path, md_path, md)

# Chunking

In [1]:
filename = 'books_text/Think_C++.md'
with open(filename, 'r', encoding='utf-8') as f:
    content = f.read()

In [2]:
import re
list_content = content.split('\n')
list_content
filtered_list = [x for x in list_content if x not in ('', '.')]
# filtered_list = [x for x in filtered_list if x.strip('.') != '']
# filtered_list = [x for x in filtered_list if x.strip().replace('.', '') != '']
filtered_list = [re.sub(r'\.\s*', '', x) for x in filtered_list]
# filtered_list = [x for x in list_content if x not in ('', '.')]
filtered_list = [x for x in filtered_list if x]

In [3]:
filtered_list

['| How to | Think | Like | a Computer | Scientist |',
 '| ------ | ----- | ---- | ---------- | --------- |',
 'C Version',
 'Thomas Scheffler',
 '|     |     | based | on previous work | by Allen BDowney |',
 '| --- | --- | ----- | ---------------- | ------------------ |',
 'Version 110',
 'June 27th, 2019',
 '2',
 '| Copyright | (C) 1999 Allen  | BDowney |     |',
 '| --------- | --------------- | --------- | --- |',
 '| Copyright | (C) 2009 Thomas | Scheffler |     |',
 'Permission is granted to copy, distribute, transmit and adapt this work un-',
 'der the Creative Commons Attribution-NonCommercial-ShareAlike 40 Inter-',
 '| national License: | https://creativecommonsorg/licenses/by-nc/40/|     |     |',
 '| ----------------- | ------------------------------------------------ | --- | --- |',
 'If you are interested in distributing a commercial version of this work, please',
 '| contact the | author(s)|               |                    |',
 '| ----------- | --------------- | -----

In [9]:
import os

MD_DIR = "books_text"

documents = []

for filename in os.listdir(MD_DIR):
    if not filename.endswith(".md"):
        continue

    filepath = os.path.join(MD_DIR, filename)

    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    lines = content.split('\n')
    cleaned_list = [x for x in lines if x not in ('', '.')]
    cleaned_list = [re.sub(r'\.\s*', '', x) for x in cleaned_list]
    cleaned_list = [x for x in cleaned_list if x]

    book = {
        "source": filename,
        "content": cleaned_list
    }

    documents.append(book)

print(documents[1])

{'source': 'Think_Python_2e.md', 'content': ['Think Python', 'How to Think Like a Computer Scientist', '2nd Edition, Version 240', '\x0c\x0cThink Python', 'How to Think Like a Computer Scientist', '2nd Edition, Version 240', 'Allen Downey', 'Green Tea Press', 'Needham, Massachusetts', '\x0cCopyright © 2015 Allen Downey', 'Green Tea Press', '9 Washburn Ave', 'Needham MA 02492', 'Permission is granted to copy, distribute, and/or modify this document under the terms of the', 'Creative Commons Attribution-NonCommercial 30 Unported License, which is available at http:', '//creativecommonsorg/licenses/by-nc/30/', 'The original form of this book is LATEX source codeCompiling this LATEX source has the effect of gen-', 'erating a device-independent representation of a textbook, which can be converted to other formats', 'and printed', 'The LATEX source for this book is available from http://wwwthinkpythoncom', '\x0cPreface', 'The strange history of this book', 'In January 1999 I was preparing to

In [10]:
len(documents)

7

In [11]:
from gitsource import chunk_documents

document_chunks = chunk_documents(documents, size=100, step=50)

In [13]:
target_name = "Think_Python"
num_chunks = 0

for doc in document_chunks:
    if target_name in doc["source"]:
        num_chunks += 1
print(f"{target_name} has {num_chunks} Chunks")

Think_Python has 194 Chunks


In [14]:
len(document_chunks)

989

In [15]:
def prepare_documents(chunks):
    for doc in chunks:
        combined_text = "\n".join(doc['content'])
        doc['content'] = combined_text
    return chunks

In [16]:
documents = prepare_documents(document_chunks)

# Indexing

In [17]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [18]:
results = index.search("python function definition", num_results=5)

In [19]:
results[0].get('source')

'Think_Python_2e.md'

# RAG

In [21]:
import json
from openai import OpenAI
openai_client = OpenAI()

In [22]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.usage, response.output_text

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    usage, answer = llm(prompt, instructions)
    return usage, answer

In [23]:
query = "python function definition"
usage, result = rag(query)

In [24]:
result

"In Python, a function is defined using the `def` keyword followed by the function name and parentheses containing any parameters. Here's the basic syntax:\n\n```python\ndef function_name(parameters):\n    # Function body\n    # Code to execute\n    return result  # Optional\n```\n\n### Example:\nHere’s a simple example of a function that adds two numbers:\n\n```python\ndef add_numbers(a, b):\n    return a + b\n\n# You can call the function like this:\nresult = add_numbers(5, 3)\nprint(result)  # Output: 8\n```\n\nIn this example:\n- `def add_numbers(a, b):` defines a function named `add_numbers` that takes two parameters, `a` and `b`.\n- The function body contains a single line of code that returns the sum of `a` and `b`.\n- You can call the function with actual values, and it returns the result.\n\nFeel free to ask if you need more information or examples!"

In [25]:
usage

ResponseUsage(input_tokens=7115, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=205, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7320)

# structured output

In [26]:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

In [27]:
instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

def llm_structured(
    user_prompt,
    output_type,
    instructions=None,
    model="gpt-4o-mini",
):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.usage, response.output_parsed

def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )

In [28]:
usage, answer = rag_structured('python function definition')

print(answer.answer[:100])
print(answer.found_answer)


In Python, a function is defined using the `def` keyword, followed by the function name and parenthe
True


In [29]:
usage

ResponseUsage(input_tokens=7317, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=274, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7591)